# Segmentação de pluma de fumaça com SAM (Segment Anything Model)

**Disciplina / atividade:** avaliar a máscara gerada pelo SAM para uma pluma de fumaça,
aprimorar código, *prompts* e apresentação da máscara, e justificar as melhorias.

---

## O que este notebook entrega

| Item pedido na atividade | Onde está |
|---|---|
| Observar a máscara gerada pelo SAM (cobre a pluma? inclui o que não é pluma? deixa fumaça de fora?) | Etapas 1, 5 e 6 |
| Testar caixa delimitadora e pontos positivos/negativos | Etapa 3 |
| **Pelo menos três** configurações diferentes de *prompts* | Etapa 3 — são **6** configurações (A–F) |
| Figura lado a lado: original, máscara binária, máscara sobreposta | Etapa 4 |
| Quais características da imagem dificultaram a segmentação | Etapas 2 e 6 (com medições, não só opinião) |
| Como iluminação, contraste e semelhança fumaça/nuvem afetaram | Etapa 6 (experimentos controlados) |
| Avaliação no próprio código | Etapa 5 (tabela de métricas) + Etapa 7 (relatório automático) |

## Como rodar

1. `Ambiente de execução` → `Alterar o tipo de ambiente de execução` → **GPU (T4)**.
   Sem GPU o notebook ainda roda, mas cai automaticamente para o modelo `vit_b` (mais leve e menos preciso).
2. `Ambiente de execução` → `Executar tudo`.
3. Para usar **sua própria imagem**: na Etapa 0, troque `CFG["IMAGEM"]` para `"upload"` e envie o arquivo.
   Depois ajuste os *prompts* na Etapa 3 (as coordenadas são **frações** da largura/altura, então
   funcionam em qualquer resolução, mas precisam apontar para a sua pluma).

> **Resumo da conclusão (detalhada na Etapa 8):** o SAM sozinho não "sabe" o que é fumaça.
> Ele segmenta a região mais *coerente* dentro do *prompt*. Como a pluma tem borda difusa,
> baixa saturação e luminância parecida com a de nuvens e da neblina/haze do horizonte, o
> *prompt* precisa dizer explicitamente o que **não** é pluma — por isso os pontos negativos
> são o fator que mais melhora o resultado neste caso.

In [ ]:
# =====================================================================
# ETAPA 0.1 — Instalação do SAM e download do checkpoint
# =====================================================================
# O SAM não vem instalado no Colab. Instalamos a partir do repositório oficial.

!pip install -q git+https://github.com/facebookresearch/segment-anything.git

import os, sys, time, json, urllib.request, textwrap
import torch

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"

# vit_h é o melhor, mas pesa 2,4 GB e é lento em CPU. Sem GPU, usamos vit_b (375 MB).
CHECKPOINTS = {
    "vit_h": ("sam_vit_h_4b8939.pth", "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"),
    "vit_l": ("sam_vit_l_0b3195.pth", "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth"),
    "vit_b": ("sam_vit_b_01ec64.pth", "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"),
}
TIPO_MODELO = "vit_h" if DISPOSITIVO == "cuda" else "vit_b"
ARQ_CKPT, URL_CKPT = CHECKPOINTS[TIPO_MODELO]


def baixar(url, destino):
    """Baixa com barra de progresso simples e não refaz download se já existir."""
    if os.path.exists(destino) and os.path.getsize(destino) > 1e6:
        print(f"[ok] {destino} já está no disco ({os.path.getsize(destino)/1e6:.0f} MB)")
        return destino
    print(f"[..] baixando {url}")
    t0 = time.time()

    def progresso(blocos, tam_bloco, total):
        if total > 0 and blocos % 200 == 0:
            pct = 100 * blocos * tam_bloco / total
            print(f"\r     {min(pct, 100):5.1f}%  ({blocos*tam_bloco/1e6:6.0f} MB)", end="")

    urllib.request.urlretrieve(url, destino, progresso)
    print(f"\r[ok] {destino} baixado em {time.time()-t0:.0f}s")
    return destino


baixar(URL_CKPT, ARQ_CKPT)

print(f"\nDispositivo : {DISPOSITIVO}")
if DISPOSITIVO == "cuda":
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
else:
    print("GPU         : (nenhuma) — as etapas automáticas ficarão lentas; considere ativar a GPU")
print(f"Modelo SAM  : {TIPO_MODELO}")

In [ ]:
# =====================================================================
# ETAPA 0.2 — Imports, configuração central e utilitários
# =====================================================================
import numpy as np
import cv2
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import pandas as pd
from dataclasses import dataclass, field
from typing import Optional, Sequence

np.random.seed(42)
torch.manual_seed(42)

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 140,
    "figure.facecolor": "white",
    "axes.titlesize": 10,
    "axes.titleweight": "bold",
    "font.size": 9,
})

PASTA_SAIDA = "saida"
os.makedirs(PASTA_SAIDA, exist_ok=True)

# ---------------------------------------------------------------------
# Painel único de configuração (tudo que vale a pena mexer está aqui)
# ---------------------------------------------------------------------
CFG = {
    # "socal" | "yellowstone" | "upload" | caminho/URL de uma imagem sua
    "IMAGEM": "socal",
    # redimensiona o maior lado para este valor (velocidade + reprodutibilidade)
    "LADO_MAX": 1280,
    # roda o SamAutomaticMaskGenerator na Etapa 1 (lento em CPU)
    "RODAR_AUTOMATICO": True,
    # resolução usada só no modo automático (ele é O(n²) em pontos)
    "LADO_AUTOMATICO": 768,
    # roda os experimentos de sensibilidade da Etapa 6 (reprocessa a imagem algumas vezes)
    "RODAR_SENSIBILIDADE": True,
    # nº de perturbações aleatórias da caixa no teste de robustez
    "N_JITTER": 6,
}


def salvar_fig(fig, nome):
    """Salva a figura em saida/ para depois exportar junto com o notebook."""
    caminho = os.path.join(PASTA_SAIDA, nome)
    fig.savefig(caminho, bbox_inches="tight", facecolor="white")
    return caminho


def para_px(pontos_frac, largura, altura):
    """Converte coordenadas fracionárias (0–1) para pixels inteiros.

    Trabalhar em fração é o que permite trocar de imagem/resolução sem
    reescrever todos os prompts.
    """
    p = np.asarray(pontos_frac, dtype=float).reshape(-1, 2)
    return np.stack([p[:, 0] * largura, p[:, 1] * altura], axis=1)


def caixa_para_px(caixa_frac, largura, altura):
    x0, y0, x1, y1 = caixa_frac
    return np.array([x0 * largura, y0 * altura, x1 * largura, y1 * altura], dtype=float)


def retangulo_para_mascara(caixa_frac, largura, altura):
    """Transforma um retângulo fracionário em máscara booleana (usado nas ROIs)."""
    x0, y0, x1, y1 = caixa_para_px(caixa_frac, largura, altura).astype(int)
    m = np.zeros((altura, largura), dtype=bool)
    m[max(y0, 0):min(y1, altura), max(x0, 0):min(x1, largura)] = True
    return m


def poligono_para_mascara(vertices_frac, largura, altura):
    pts = para_px(vertices_frac, largura, altura).astype(np.int32)
    m = np.zeros((altura, largura), dtype=np.uint8)
    cv2.fillPoly(m, [pts], 1)
    return m.astype(bool)


print("Configuração carregada.")

---
# Etapa 0.3 — A imagem

Duas imagens de domínio público (Wikimedia Commons) já vêm configuradas, com *prompts* calibrados:

* **`socal`** — incêndio no sul da Califórnia (2007). Tem **os três distratores citados no enunciado
  ao mesmo tempo**: céu azul, casas/telhados e uma faixa de *haze* esbranquiçada no horizonte que é
  quase indistinguível da base da pluma. Além disso, uma placa de trânsito **oclui parte da pluma** —
  um caso ótimo para discutir falha de continuidade.
* **`yellowstone`** — pluma com **pirocúmulos**: a fumaça gera nuvens de verdade no topo. É o caso
  extremo de "semelhança entre fumaça e nuvem", porque nem um humano consegue traçar a fronteira
  com segurança.

Para usar a sua própria imagem, ponha `CFG["IMAGEM"] = "upload"` (ou um caminho/URL) e ajuste os
*prompts* da Etapa 3 — a Etapa 0.4 imprime a imagem com uma grade de coordenadas justamente para isso.

In [ ]:
# =====================================================================
# ETAPA 0.4 — Catálogo de imagens, carregamento e grade de coordenadas
# =====================================================================

CATALOGO = {
    "socal": {
        "url": ("https://commons.wikimedia.org/wiki/Special:FilePath/"
                "Smoke%20plume%20from%20SoCal%20wildfire%20October%202007.jpg?width=1280"),
        "credito": "Smoke plume from SoCal wildfire, outubro/2007 — Wikimedia Commons",
        # ---- prompts em coordenadas FRACIONÁRIAS (x/larg, y/alt) ----
        "caixa": (0.289, 0.266, 0.999, 0.625),
        "positivos": [(0.336, 0.479), (0.484, 0.438), (0.625, 0.385),
                      (0.742, 0.344), (0.320, 0.531)],
        "negativos": [(0.117, 0.156),   # céu azul, canto superior esquerdo
                      (0.547, 0.156),   # céu acima da pluma
                      (0.781, 0.125),   # céu à direita
                      (0.234, 0.583),   # cordilheira / encosta
                      (0.117, 0.625),   # telhados das casas à esquerda
                      (0.570, 0.635),   # telhado da casa central
                      (0.469, 0.833),   # asfalto
                      (0.883, 0.271)],  # placa "No Stopping" que oclui a pluma
        # ROIs retangulares para diagnóstico e medição de vazamento
        "rois": {
            "pluma":      (0.45, 0.33, 0.62, 0.45),
            "ceu":        (0.05, 0.05, 0.30, 0.20),
            "nuvem_haze": (0.60, 0.50, 0.78, 0.56),
            "edificacao": (0.02, 0.56, 0.22, 0.66),
        },
        # referência manual APROXIMADA (ver aviso na Etapa 5)
        "referencia": [(0.305, 0.552), (0.316, 0.490), (0.328, 0.448), (0.352, 0.425),
                       (0.391, 0.420), (0.426, 0.442), (0.438, 0.406), (0.469, 0.375),
                       (0.516, 0.359), (0.563, 0.359), (0.609, 0.344), (0.648, 0.313),
                       (0.688, 0.297), (0.727, 0.286), (0.773, 0.292), (0.820, 0.302),
                       (0.875, 0.302), (0.938, 0.297), (0.996, 0.302), (0.998, 0.392),
                       (0.930, 0.460), (0.855, 0.500), (0.781, 0.530), (0.703, 0.545),
                       (0.625, 0.540), (0.547, 0.526), (0.484, 0.516), (0.430, 0.521),
                       (0.359, 0.536)],
    },
    "yellowstone": {
        "url": ("https://commons.wikimedia.org/wiki/Special:FilePath/"
                "Wildfire%20in%20Yellowstone%20National%20Park%20produces%20Pyrocumulus%20clouds1.jpg?width=1280"),
        "credito": "Wildfire in Yellowstone NP produces Pyrocumulus clouds — Wikimedia Commons",
        "caixa": (0.641, 0.026, 0.999, 0.383),
        "positivos": [(0.750, 0.115), (0.805, 0.191), (0.898, 0.204),
                      (0.703, 0.255), (0.961, 0.293)],
        "negativos": [(0.234, 0.115),   # nuvem branca (cúmulo "normal") à esquerda
                      (0.391, 0.102),   # outra nuvem branca
                      (0.047, 0.038),   # céu azul
                      (0.961, 0.051),   # céu azul à direita
                      (0.469, 0.510),   # vegetação / campo
                      (0.156, 0.319)],  # camada difusa de fumaça distante
        "rois": {
            "pluma":      (0.72, 0.15, 0.86, 0.26),
            "ceu":        (0.00, 0.00, 0.08, 0.06),
            "nuvem_haze": (0.18, 0.07, 0.34, 0.16),
            "edificacao": (0.30, 0.50, 0.60, 0.60),   # aqui: vegetação/terreno
        },
        "referencia": None,
    },
}


# Prompts genéricos usados quando a imagem é do próprio aluno: servem só para o notebook
# rodar de ponta a ponta sem quebrar. Eles PRECISAM ser ajustados na Etapa 3.
PRESET_PADRAO = {
    "caixa": (0.25, 0.20, 0.95, 0.60),
    "positivos": [(0.45, 0.35), (0.55, 0.40), (0.65, 0.35), (0.50, 0.45), (0.60, 0.30)],
    "negativos": [(0.10, 0.10), (0.90, 0.10), (0.50, 0.90), (0.10, 0.85)],
    "rois": {
        "pluma":      (0.45, 0.32, 0.60, 0.44),
        "ceu":        (0.05, 0.05, 0.25, 0.18),
        "nuvem_haze": (0.62, 0.50, 0.86, 0.58),
        "edificacao": (0.05, 0.78, 0.30, 0.92),
    },
    "referencia": None,
    "credito": "imagem do usuário",
}


def carregar_imagem(chave):
    """Retorna (imagem_rgb, preset). Aceita chave do catálogo, caminho, URL ou 'upload'."""
    preset = {k: (dict(v) if isinstance(v, dict) else v) for k, v in PRESET_PADRAO.items()}
    if chave in CATALOGO:
        preset = dict(CATALOGO[chave])
        destino = f"{chave}.jpg"
        if not os.path.exists(destino):
            req = urllib.request.Request(preset["url"], headers={"User-Agent": "colab-sam-atividade/1.0"})
            with urllib.request.urlopen(req) as r, open(destino, "wb") as f:
                f.write(r.read())
        caminho = destino
    elif chave == "upload":
        from google.colab import files
        enviados = files.upload()
        caminho = list(enviados.keys())[0]
    elif str(chave).startswith("http"):
        caminho = "imagem_usuario.jpg"
        req = urllib.request.Request(chave, headers={"User-Agent": "colab-sam-atividade/1.0"})
        with urllib.request.urlopen(req) as r, open(caminho, "wb") as f:
            f.write(r.read())
    else:
        caminho = chave

    bgr = cv2.imread(caminho)
    if bgr is None:
        raise FileNotFoundError(f"Não consegui ler a imagem: {caminho}")
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    # padroniza o tamanho: velocidade e comparabilidade entre execuções
    h, w = rgb.shape[:2]
    escala = CFG["LADO_MAX"] / max(h, w)
    if escala < 1.0:
        rgb = cv2.resize(rgb, (round(w * escala), round(h * escala)), interpolation=cv2.INTER_AREA)
    return rgb, preset


def mostrar_com_grade(img, titulo="Imagem com grade de coordenadas", passo=100, salvar=None):
    """Mostra a imagem com uma grade rotulada — serve para o aluno ler coordenadas
    de prompts na mão, sem depender de clique interativo (que quebra fácil no Colab)."""
    h, w = img.shape[:2]
    fig, ax = plt.subplots(figsize=(11, 11 * h / w))
    ax.imshow(img)
    for x in range(0, w, passo):
        ax.axvline(x, color="yellow", lw=0.4, alpha=0.8)
        ax.text(x + 2, 14, f"{x}\n{x/w:.2f}", color="red", fontsize=6, va="top")
    for y in range(0, h, passo):
        ax.axhline(y, color="yellow", lw=0.4, alpha=0.8)
        ax.text(2, y + 4, f"{y} / {y/h:.2f}", color="red", fontsize=6, va="top")
    ax.set_title(f"{titulo}  —  {w}x{h}px  (rótulos: pixel / fração)")
    ax.axis("off")
    fig.tight_layout()
    if salvar:
        salvar_fig(fig, salvar)
    plt.show()


IMG, PRESET = carregar_imagem(CFG["IMAGEM"])
ALT, LARG = IMG.shape[:2]
print(f"Imagem: {CFG['IMAGEM']}  |  {LARG}x{ALT} px  |  {PRESET['credito']}")
if CFG["IMAGEM"] not in CATALOGO:
    print("\n*** ATENÇÃO: imagem sem prompts calibrados. Os valores de PRESET_PADRAO são\n"
          "    genéricos e quase certamente NÃO apontam para a sua pluma. Use a grade de\n"
          "    coordenadas abaixo para reescrever 'caixa', 'positivos', 'negativos' e as\n"
          "    'rois' (basta editar PRESET depois desta célula). ***\n")
mostrar_com_grade(IMG, salvar="00_imagem_com_grade.png")

---
# Etapa 1 — Diagnóstico do ponto de partida: SAM no modo automático

O `SamAutomaticMaskGenerator` varre a imagem com uma grade de pontos e devolve **tudo** o que ele
considera um objeto coerente. É exatamente o cenário descrito no enunciado — "a segmentação
automática pode incluir partes do céu, nuvens ou edifícios".

O que medimos aqui (em vez de só olhar):

1. **quantas** máscaras o modelo produz;
2. **quantas delas encostam na pluma** → se for mais de uma, a pluma foi *fragmentada*;
3. **quanto do céu** (ROI de céu puro) foi coberto por alguma máscara;
4. se existe **alguma máscara única** que sozinha corresponda à pluma inteira.

Esse é o "antes" contra o qual as configurações de *prompt* da Etapa 3 serão comparadas.

In [ ]:
# =====================================================================
# ETAPA 1 — Modo automático (baseline sem prompt)
# =====================================================================
from segment_anything import sam_model_registry, SamPredictor, SamAutomaticMaskGenerator

t0 = time.time()
sam = sam_model_registry[TIPO_MODELO](checkpoint=ARQ_CKPT).to(DISPOSITIVO)
print(f"Modelo carregado em {time.time()-t0:.1f}s")


def mostrar_anotacoes(ax, anotacoes, alpha=0.55):
    """Pinta cada máscara com uma cor aleatória, das maiores para as menores."""
    if not anotacoes:
        return
    ordenadas = sorted(anotacoes, key=lambda a: a["area"], reverse=True)
    h, w = ordenadas[0]["segmentation"].shape
    camada = np.zeros((h, w, 4), dtype=float)
    rng = np.random.default_rng(0)
    for a in ordenadas:
        camada[a["segmentation"]] = np.concatenate([rng.random(3), [alpha]])
    ax.imshow(camada)


RESUMO_AUTOMATICO = {}

if CFG["RODAR_AUTOMATICO"]:
    # roda numa versão reduzida: o custo cresce com o nº de pontos da grade
    esc_auto = CFG["LADO_AUTOMATICO"] / max(ALT, LARG)
    img_auto = cv2.resize(IMG, (round(LARG * esc_auto), round(ALT * esc_auto)),
                          interpolation=cv2.INTER_AREA) if esc_auto < 1 else IMG.copy()
    h_a, w_a = img_auto.shape[:2]

    gerador = SamAutomaticMaskGenerator(
        model=sam,
        points_per_side=24,          # 24x24 = 576 pontos; 32 (padrão) é mais lento
        pred_iou_thresh=0.86,
        stability_score_thresh=0.92,
        min_mask_region_area=200,
    )
    t0 = time.time()
    anotacoes = gerador.generate(img_auto)
    print(f"{len(anotacoes)} máscaras automáticas em {time.time()-t0:.0f}s")

    # --- medições sobre o resultado automático ---
    caixa_pluma_auto = caixa_para_px(PRESET["caixa"], w_a, h_a).astype(int) if PRESET["caixa"] else None
    roi_pluma = retangulo_para_mascara(PRESET["rois"]["pluma"], w_a, h_a) if PRESET["rois"] else None
    roi_ceu = retangulo_para_mascara(PRESET["rois"]["ceu"], w_a, h_a) if PRESET["rois"] else None

    tocam_pluma, melhor_iou_caixa, melhor_mascara = 0, 0.0, None
    cobertura_ceu = np.zeros((h_a, w_a), dtype=bool)
    if caixa_pluma_auto is not None:
        mascara_caixa = np.zeros((h_a, w_a), dtype=bool)
        x0, y0, x1, y1 = caixa_pluma_auto
        mascara_caixa[y0:y1, x0:x1] = True
        for a in anotacoes:
            seg = a["segmentation"]
            if roi_pluma is not None and (seg & roi_pluma).sum() > 0.02 * roi_pluma.sum():
                tocam_pluma += 1
            inter = (seg & mascara_caixa).sum()
            uniao = (seg | mascara_caixa).sum()
            if uniao and inter / uniao > melhor_iou_caixa:
                melhor_iou_caixa, melhor_mascara = inter / uniao, seg
            cobertura_ceu |= seg

    pct_ceu = 100 * (cobertura_ceu & roi_ceu).sum() / roi_ceu.sum() if roi_ceu is not None else float("nan")
    RESUMO_AUTOMATICO = {
        "n_mascaras": len(anotacoes),
        "mascaras_tocando_a_pluma": tocam_pluma,
        "melhor_iou_contra_a_caixa": melhor_iou_caixa,
        "pct_da_roi_de_ceu_coberta": pct_ceu,
    }

    fig, axs = plt.subplots(1, 3, figsize=(16, 5))
    axs[0].imshow(img_auto); axs[0].set_title("Original")
    axs[1].imshow(img_auto); mostrar_anotacoes(axs[1], anotacoes)
    axs[1].set_title(f"SamAutomaticMaskGenerator — {len(anotacoes)} máscaras")
    axs[2].imshow(img_auto)
    if melhor_mascara is not None:
        axs[2].imshow(np.dstack([melhor_mascara * 1.0, melhor_mascara * 0.2,
                                 melhor_mascara * 0.2, melhor_mascara * 0.55]))
    if caixa_pluma_auto is not None:
        x0, y0, x1, y1 = caixa_pluma_auto
        axs[2].add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="yellow", lw=2))
    axs[2].set_title(f"Máscara automática mais parecida com a pluma\nIoU vs. caixa = {melhor_iou_caixa:.2f}")
    for a in axs:
        a.axis("off")
    fig.suptitle("Etapa 1 — sem prompt, o SAM segmenta a cena inteira, não a pluma", y=1.02)
    fig.tight_layout(); salvar_fig(fig, "01_baseline_automatico.png"); plt.show()

    print(json.dumps(RESUMO_AUTOMATICO, indent=2, ensure_ascii=False, default=float))
else:
    print("Modo automático desativado em CFG['RODAR_AUTOMATICO'].")

---
# Etapa 2 — Por que esta imagem é difícil: diagnóstico fotométrico

Antes de culpar o modelo, vale medir a imagem. A pergunta do enunciado — *"como a iluminação, o
contraste e a semelhança entre fumaça e nuvens afetaram o resultado?"* — pode ser respondida com
números, comparando quatro regiões de amostra (ROIs): **pluma**, **céu**, **nuvem/haze** e
**edificação/terreno**.

Três medidas, cada uma ligada a um mecanismo de falha:

* **Contraste RMS local** — o SAM é um modelo visual; onde o gradiente é fraco, a borda prevista
  fica indefinida. Fumaça tem borda difusa *por natureza física* (é um meio semitransparente,
  não um objeto com superfície).
* **Saturação (HSV)** — fumaça e nuvem são ambas quase acromáticas. Se a saturação das duas for
  parecida, cor não ajuda a separar.
* **Distância de Bhattacharyya entre histogramas** — quantifica "o quanto as distribuições se
  sobrepõem". Valor perto de 0 = regiões estatisticamente indistinguíveis.
  A comparação decisiva é `pluma × nuvem/haze` contra `pluma × céu`.

In [ ]:
# =====================================================================
# ETAPA 2 — Diagnóstico fotométrico da imagem
# =====================================================================

def amostras_roi(img_rgb, roi_frac):
    """Extrai, para uma ROI, as features usadas no diagnóstico."""
    h, w = img_rgb.shape[:2]
    m = retangulo_para_mascara(roi_frac, w, h)
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    cinza = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    gx = cv2.Sobel(cinza, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(cinza, cv2.CV_32F, 0, 1, ksize=3)
    grad = np.sqrt(gx ** 2 + gy ** 2)
    media = cv2.blur(cinza.astype(np.float32), (9, 9))
    desvio = np.sqrt(np.maximum(cv2.blur(cinza.astype(np.float32) ** 2, (9, 9)) - media ** 2, 0))
    return {
        "luminancia": cinza[m].astype(float),
        "saturacao": hsv[:, :, 1][m].astype(float),
        "gradiente": grad[m],
        "textura": desvio[m],
    }


LIMITE_BHAT = 8.0  # saturação: acima disso as distribuições simplesmente não se tocam


def bhattacharyya(a, b, bins=64, faixa=None):
    """Distância de Bhattacharyya entre dois conjuntos de amostras.

    0   = distribuições idênticas (regiões indistinguíveis por essa feature)
    ~1  = razoavelmente separáveis
    8.0 = valor de saturação usado aqui: histogramas sem nenhuma sobreposição
          (a distância verdadeira seria infinita, então truncamos para a tabela
          continuar legível).
    """
    faixa = faixa or (float(min(a.min(), b.min())), float(max(a.max(), b.max())) + 1e-6)
    ha, _ = np.histogram(a, bins=bins, range=faixa)
    hb, _ = np.histogram(b, bins=bins, range=faixa)
    ha = ha / (ha.sum() + 1e-12)
    hb = hb / (hb.sum() + 1e-12)
    return float(min(-np.log(np.sum(np.sqrt(ha * hb)) + 1e-12), LIMITE_BHAT))


def razao_fisher(a, b):
    """Separabilidade de Fisher: (dif. das médias)² / (soma das variâncias)."""
    return float((a.mean() - b.mean()) ** 2 / (a.var() + b.var() + 1e-9))


AMOSTRAS = {nome: amostras_roi(IMG, roi) for nome, roi in PRESET["rois"].items()}

# ---- tabela de estatísticas por região ----
linhas = []
for nome, am in AMOSTRAS.items():
    linhas.append({
        "região": nome,
        "luminância média": am["luminancia"].mean(),
        "contraste RMS": am["luminancia"].std(),
        "saturação média": am["saturacao"].mean(),
        "gradiente médio": am["gradiente"].mean(),
        "textura (σ local)": am["textura"].mean(),
    })
TAB_REGIOES = pd.DataFrame(linhas).set_index("região").round(2)

# ---- separabilidade pluma × resto ----
sep = []
for outro in [n for n in AMOSTRAS if n != "pluma"]:
    for feat in ["luminancia", "saturacao", "gradiente", "textura"]:
        sep.append({
            "comparação": f"pluma × {outro}",
            "feature": feat,
            "Bhattacharyya": bhattacharyya(AMOSTRAS["pluma"][feat], AMOSTRAS[outro][feat]),
            "Fisher": razao_fisher(AMOSTRAS["pluma"][feat], AMOSTRAS[outro][feat]),
        })
TAB_SEPARABILIDADE = (pd.DataFrame(sep)
                      .pivot(index="comparação", columns="feature", values="Bhattacharyya")
                      .round(3))

print("Estatísticas por região:")
display(TAB_REGIOES)
print("\nDistância de Bhattacharyya (0 = indistinguível; quanto maior, mais separável):")
display(TAB_SEPARABILIDADE)

# ---- figura de diagnóstico ----
hsv = cv2.cvtColor(IMG, cv2.COLOR_RGB2HSV)
cinza = cv2.cvtColor(IMG, cv2.COLOR_RGB2GRAY)
gx = cv2.Sobel(cinza, cv2.CV_32F, 1, 0, ksize=3)
gy = cv2.Sobel(cinza, cv2.CV_32F, 0, 1, ksize=3)
GRAD = np.sqrt(gx ** 2 + gy ** 2)

fig, axs = plt.subplots(2, 3, figsize=(16, 9))
cores_roi = {"pluma": "lime", "ceu": "deepskyblue", "nuvem_haze": "orange", "edificacao": "magenta"}

axs[0, 0].imshow(IMG)
for nome, roi in PRESET["rois"].items():
    x0, y0, x1, y1 = caixa_para_px(roi, LARG, ALT)
    axs[0, 0].add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False,
                                  ec=cores_roi.get(nome, "white"), lw=2))
    axs[0, 0].text(x0, y0 - 4, nome, color=cores_roi.get(nome, "white"), fontsize=8, weight="bold")
axs[0, 0].set_title("ROIs de amostragem")

axs[0, 1].imshow(hsv[:, :, 1], cmap="viridis"); axs[0, 1].set_title("Saturação (HSV)\nfumaça e nuvem: ambas baixas")
axs[0, 2].imshow(np.clip(GRAD, 0, 120), cmap="magma"); axs[0, 2].set_title("Magnitude do gradiente\nborda da pluma quase some")

for feat, ax, titulo in [("luminancia", axs[1, 0], "Luminância"),
                         ("saturacao", axs[1, 1], "Saturação"),
                         ("gradiente", axs[1, 2], "Gradiente")]:
    for nome, am in AMOSTRAS.items():
        ax.hist(am[feat], bins=50, alpha=0.5, density=True,
                label=nome, color=cores_roi.get(nome, None))
    ax.set_title(f"Histograma — {titulo}")
    if feat == "gradiente":
        ax.set_xlim(0, 200)   # a cauda longa esconde a diferença entre as regiões
    ax.legend(fontsize=7)

for a in axs[0]:
    a.axis("off")
fig.suptitle("Etapa 2 — o que, na própria imagem, torna a segmentação ambígua", y=1.0)
fig.tight_layout(); salvar_fig(fig, "02_diagnostico_imagem.png"); plt.show()

---
# Etapa 3 — Prompts: caixa, pontos positivos e pontos negativos

Seis configurações, em ordem crescente de informação dada ao modelo. As três primeiras já atendem
ao mínimo pedido ("pelo menos três configurações"); as três últimas são as **melhorias propostas**.

| | Configuração | *Prompt* | Hipótese que está sendo testada |
|---|---|---|---|
| **A** | Só caixa | `box` | O retângulo basta? Ele contém muito céu junto. |
| **B** | Só pontos positivos | 3 pontos `+` | Sem limite espacial, até onde o SAM "espalha" a região? |
| **C** | Caixa + positivos | `box` + 5 `+` | Os positivos corrigem a escolha de escala dentro da caixa. |
| **D** | Caixa + positivos + **negativos** | `box` + 5 `+` + 8 `−` | Dizer o que **não** é pluma (céu, nuvem, telhado, placa) resolve a ambiguidade? |
| **E** | D + **refinamento iterativo** | D, reinjetando `mask_input` | Realimentar os *logits* da predição anterior estabiliza a borda? |
| **F** | E + **pós-processamento + máscara suave** | E + morfologia + filtro guiado | Melhora a *apresentação* sem inventar região nova? |

### Duas decisões de projeto que valem ser justificadas

**1. Coordenadas em fração, não em pixel.** Todo *prompt* é `(x/largura, y/altura)`. Assim o mesmo
conjunto de *prompts* vale para a imagem redimensionada, para o modo automático (que roda menor) e
para os testes de sensibilidade — sem risco de um ponto "escorregar" para fora do objeto.

**2. `multimask_output=True` + escolha pelo score.** Para um *prompt* ambíguo o SAM devolve três
máscaras em escalas diferentes (parte / sub-objeto / objeto inteiro). A célula abaixo mostra as três
para o *prompt* A — é a demonstração mais direta de que **a ambiguidade é do problema, não um bug**:
as três são respostas defensáveis para "o que está dentro desta caixa".

In [ ]:
# =====================================================================
# ETAPA 3.1 — Prompts, predição e pós-processamento
# =====================================================================
predictor = SamPredictor(sam)
t0 = time.time()
predictor.set_image(IMG)
print(f"Embedding da imagem calculado em {time.time()-t0:.1f}s")

CAIXA_PX = caixa_para_px(PRESET["caixa"], LARG, ALT)
POS_PX = para_px(PRESET["positivos"], LARG, ALT)
NEG_PX = para_px(PRESET["negativos"], LARG, ALT)
ROIS = {nome: retangulo_para_mascara(roi, LARG, ALT) for nome, roi in PRESET["rois"].items()}
REFERENCIA = (poligono_para_mascara(PRESET["referencia"], LARG, ALT)
              if PRESET.get("referencia") else None)


def prever(box=None, pontos=None, rotulos=None, mask_input=None, multimask=True, pred=None):
    """Envelopa predictor.predict com uma interface uniforme.

    Retorna (melhor_mascara, melhor_score, logits_da_melhor, todas_mascaras, todos_scores).
    Quando `mask_input` é usado, multimask é forçado a False (recomendação do próprio SAM:
    o refinamento parte de uma hipótese já escolhida, não reabre a ambiguidade).
    """
    pred = pred or predictor
    kw = {}
    if pontos is not None and len(pontos) > 0:
        kw["point_coords"] = np.asarray(pontos, dtype=float)
        kw["point_labels"] = np.asarray(rotulos, dtype=int)
    if box is not None:
        kw["box"] = np.asarray(box, dtype=float)[None, :]
    if mask_input is not None:
        kw["mask_input"] = mask_input
        multimask = False
    mascaras, scores, logits = pred.predict(multimask_output=multimask, **kw)
    i = int(np.argmax(scores))
    return mascaras[i].astype(bool), float(scores[i]), logits[i][None, :, :], mascaras, scores


def pos_processar(mask, abrir=5, fechar=11, manter_maior=True, min_area_frac=0.0008):
    """Limpeza morfológica: tira ilhas espúrias e costura buracos finos da borda.

    Justificativa: o SAM decide pixel a pixel a partir de um mapa de baixa resolução (256x256);
    os respingos isolados que sobram quase nunca são pluma — são céu com luminância parecida.
    """
    m = (mask.astype(np.uint8) * 255)
    if abrir > 0:
        m = cv2.morphologyEx(m, cv2.MORPH_OPEN,
                             cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (abrir, abrir)))
    if fechar > 0:
        m = cv2.morphologyEx(m, cv2.MORPH_CLOSE,
                             cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (fechar, fechar)))
    b = (m > 0).astype(np.uint8)
    n, rot, stats, _ = cv2.connectedComponentsWithStats(b, 8)
    if n > 1:
        areas = stats[1:, cv2.CC_STAT_AREA]
        if manter_maior:
            manter = [1 + int(np.argmax(areas))]
        else:
            manter = [1 + i for i, a in enumerate(areas) if a >= min_area_frac * b.size]
        b = np.isin(rot, manter).astype(np.uint8)
    return b.astype(bool)


def filtro_guiado(guia_cinza, p, raio=14, eps=1e-3):
    """Guided filter (He et al., 2010) em ~8 linhas de numpy/OpenCV.

    Usado para gerar uma máscara SUAVE: a fumaça é semitransparente, então uma borda
    binária é fisicamente errada — existe uma faixa de mistura fumaça/fundo.
    """
    I = guia_cinza.astype(np.float32) / 255.0
    p = p.astype(np.float32)
    k = (raio, raio)
    mI, mp = cv2.boxFilter(I, -1, k), cv2.boxFilter(p, -1, k)
    cov = cv2.boxFilter(I * p, -1, k) - mI * mp
    var = cv2.boxFilter(I * I, -1, k) - mI * mI
    a = cov / (var + eps)
    b = mp - a * mI
    return np.clip(cv2.boxFilter(a, -1, k) * I + cv2.boxFilter(b, -1, k), 0, 1)


# ---- demonstração da ambiguidade: as 3 saídas do multimask para a caixa sozinha ----
_, _, _, tres_mascaras, tres_scores = prever(box=CAIXA_PX, multimask=True)
fig, axs = plt.subplots(1, 3, figsize=(16, 4.4))
for i, ax in enumerate(axs):
    ax.imshow(IMG)
    ax.imshow(np.dstack([tres_mascaras[i] * 1.0, tres_mascaras[i] * 0.25,
                         tres_mascaras[i] * 0.25, tres_mascaras[i] * 0.5]))
    x0, y0, x1, y1 = CAIXA_PX
    ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="yellow", lw=2))
    ax.set_title(f"candidata {i+1} — score previsto {tres_scores[i]:.3f}\n"
                 f"área = {100*tres_mascaras[i].mean():.1f}% da imagem")
    ax.axis("off")
fig.suptitle("Etapa 3 — com a caixa sozinha o SAM devolve 3 respostas plausíveis (ambiguidade de escala)", y=1.04)
fig.tight_layout(); salvar_fig(fig, "03_ambiguidade_multimask.png"); plt.show()

In [ ]:
# =====================================================================
# ETAPA 3.2 — As seis configurações de prompt
# =====================================================================

CONFIGS = {
    "A — só caixa": dict(
        box=CAIXA_PX, pontos=None, rotulos=None, refinar=0, pos=False,
        descricao="Caixa delimitadora da pluma, sem nenhum ponto."),
    "B — só pontos (+)": dict(
        box=None, pontos=POS_PX[:3], rotulos=np.ones(len(POS_PX[:3]), int), refinar=0, pos=False,
        descricao="Três pontos positivos sobre o corpo da pluma, sem caixa."),
    "C — caixa + (+)": dict(
        box=CAIXA_PX, pontos=POS_PX, rotulos=np.ones(len(POS_PX), int), refinar=0, pos=False,
        descricao="Caixa mais cinco pontos positivos distribuídos ao longo da pluma."),
    "D — caixa + (+) + (−)": dict(
        box=CAIXA_PX,
        pontos=np.vstack([POS_PX, NEG_PX]),
        rotulos=np.concatenate([np.ones(len(POS_PX), int), np.zeros(len(NEG_PX), int)]),
        refinar=0, pos=False,
        descricao="D acrescenta pontos negativos em céu, nuvem/haze, telhado, encosta e placa."),
    "E — D + refinamento": dict(
        box=CAIXA_PX,
        pontos=np.vstack([POS_PX, NEG_PX]),
        rotulos=np.concatenate([np.ones(len(POS_PX), int), np.zeros(len(NEG_PX), int)]),
        refinar=2, pos=False,
        descricao="Duas iterações extras realimentando os logits em mask_input."),
    "F — E + pós-proc.": dict(
        box=CAIXA_PX,
        pontos=np.vstack([POS_PX, NEG_PX]),
        rotulos=np.concatenate([np.ones(len(POS_PX), int), np.zeros(len(NEG_PX), int)]),
        refinar=2, pos=True,
        descricao="E mais morfologia, maior componente conexo e máscara suave (filtro guiado)."),
}


def executar(cfg):
    """Roda uma configuração completa e devolve máscara, score e tempo."""
    t0 = time.time()
    mask, score, logits, _, _ = prever(box=cfg["box"], pontos=cfg["pontos"],
                                       rotulos=cfg["rotulos"], multimask=True)
    for _ in range(cfg["refinar"]):
        mask, score, logits, _, _ = prever(box=cfg["box"], pontos=cfg["pontos"],
                                           rotulos=cfg["rotulos"], mask_input=logits)
    if cfg["pos"]:
        mask = pos_processar(mask)
    return {"mascara": mask, "score": score, "segundos": time.time() - t0, "logits": logits}


RESULTADOS = {}
for nome, cfg in CONFIGS.items():
    RESULTADOS[nome] = executar(cfg)
    r = RESULTADOS[nome]
    print(f"{nome:24s}  score={r['score']:.3f}  área={100*r['mascara'].mean():5.2f}%  "
          f"({r['segundos']:.2f}s)")

# máscara suave só faz sentido para a configuração final
CINZA = cv2.cvtColor(IMG, cv2.COLOR_RGB2GRAY)
MASCARA_SUAVE = filtro_guiado(CINZA, RESULTADOS["F — E + pós-proc."]["mascara"].astype(np.float32))

---
# Etapa 4 — Apresentação da máscara

Três painéis por configuração, como pede o enunciado: **original (com os *prompts* desenhados)**,
**máscara binária** e **máscara sobreposta**.

Escolhas de apresentação, e o motivo de cada uma:

* os *prompts* aparecem **sobre a imagem original** — sem isso, o leitor não consegue julgar se o
  erro veio do modelo ou de um ponto mal colocado;
* a sobreposição usa **transparência + contorno**: só o preenchimento esconde justamente a borda,
  que é onde está o erro interessante;
* no painel binário vai o **percentual de área**, para dar escala ao que se está vendo;
* no fim, a versão **suave** (canal alfa contínuo) ao lado da binária — a comparação deixa explícito
  que o limite "duro" é uma simplificação imposta por nós, não uma propriedade da fumaça.

In [ ]:
# =====================================================================
# ETAPA 4 — Visualização lado a lado
# =====================================================================

def desenhar_prompts(ax, cfg):
    if cfg.get("box") is not None:
        x0, y0, x1, y1 = cfg["box"]
        ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False,
                               ec="yellow", lw=2.2, label="caixa"))
    if cfg.get("pontos") is not None and len(cfg["pontos"]) > 0:
        p, r = np.asarray(cfg["pontos"]), np.asarray(cfg["rotulos"])
        if (r == 1).any():
            ax.scatter(p[r == 1, 0], p[r == 1, 1], c="lime", marker="*", s=180,
                       ec="black", lw=0.7, label="positivo (+)", zorder=5)
        if (r == 0).any():
            ax.scatter(p[r == 0, 0], p[r == 0, 1], c="red", marker="X", s=110,
                       ec="black", lw=0.7, label="negativo (−)", zorder=5)


def sobrepor(img, mask, cor=(255, 60, 60), alpha=0.45, contorno=True):
    out = img.astype(np.float32).copy()
    m = mask.astype(bool)
    out[m] = (1 - alpha) * out[m] + alpha * np.array(cor, dtype=np.float32)
    out = out.astype(np.uint8)
    if contorno:
        cont, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(out, cont, -1, (255, 230, 0), 2)
    return out


def figura_tripla(nome, cfg, resultado, salvar=None):
    mask = resultado["mascara"]
    fig, axs = plt.subplots(1, 3, figsize=(16, 16 * ALT / LARG / 2.6))
    axs[0].imshow(IMG); desenhar_prompts(axs[0], cfg)
    axs[0].set_title(f"Original + prompts\n{nome}")
    axs[0].legend(loc="lower right", fontsize=7, framealpha=0.85)
    axs[1].imshow(mask, cmap="gray")
    axs[1].set_title(f"Máscara binária\nárea = {100*mask.mean():.2f}% da imagem")
    axs[2].imshow(sobrepor(IMG, mask))
    axs[2].set_title(f"Sobreposição (α=0.45 + contorno)\nscore previsto pelo SAM = {resultado['score']:.3f}")
    for a in axs:
        a.axis("off")
    fig.tight_layout()
    if salvar:
        salvar_fig(fig, salvar)
    plt.show()


for i, (nome, cfg) in enumerate(CONFIGS.items(), start=1):
    figura_tripla(nome, cfg, RESULTADOS[nome], salvar=f"04_{i}_{nome.split(' ')[0]}.png")

# ---- grade comparativa: todas as configurações numa figura só ----
n = len(CONFIGS)
fig, axs = plt.subplots(n, 3, figsize=(13, 3.1 * n))
for linha, (nome, cfg) in enumerate(CONFIGS.items()):
    mask = RESULTADOS[nome]["mascara"]
    axs[linha, 0].imshow(IMG); desenhar_prompts(axs[linha, 0], cfg)
    axs[linha, 1].imshow(mask, cmap="gray")
    axs[linha, 2].imshow(sobrepor(IMG, mask))
    axs[linha, 0].set_ylabel(nome, fontsize=9)
    if linha == 0:
        for c, t in enumerate(["original + prompts", "máscara binária", "sobreposição"]):
            axs[0, c].set_title(t)
    for a in axs[linha]:
        a.set_xticks([]); a.set_yticks([])
fig.suptitle("Etapa 4 — comparação das seis configurações de prompt", y=1.002)
fig.tight_layout(); salvar_fig(fig, "04_grade_comparativa.png"); plt.show()

# ---- binária x suave ----
melhor = RESULTADOS["F — E + pós-proc."]["mascara"]
fig, axs = plt.subplots(1, 3, figsize=(16, 16 * ALT / LARG / 2.6))
axs[0].imshow(melhor, cmap="gray"); axs[0].set_title("Máscara binária (F)")
axs[1].imshow(MASCARA_SUAVE, cmap="gray"); axs[1].set_title("Máscara suave — alfa contínuo\n(filtro guiado pela imagem)")
alfa = MASCARA_SUAVE[..., None]
comp = (IMG * (1 - 0.5 * alfa) + np.array([255, 60, 60]) * 0.5 * alfa).astype(np.uint8)
axs[2].imshow(comp); axs[2].set_title("Sobreposição com alfa contínuo\nborda difusa preservada")
for a in axs:
    a.axis("off")
fig.suptitle("Apresentação: por que uma máscara binária subestima a incerteza da borda", y=1.03)
fig.tight_layout(); salvar_fig(fig, "04_mascara_suave.png"); plt.show()

---
# Etapa 5 — Avaliação quantitativa

As três perguntas do enunciado viram três métricas:

| Pergunta | Métrica |
|---|---|
| *A máscara cobre a pluma?* | `recall` contra a referência e `% da ROI de pluma coberta` |
| *Inclui regiões que não pertencem a ela?* | `vazamento`: % das ROIs de céu / nuvem-haze / edificação que a máscara invadiu, e `precision` |
| *Deixa partes da fumaça de fora?* | `1 − recall`, e o mapa de erro (falso negativo em azul) |

Métricas de forma complementares: `nº de componentes conexos` (fragmentação) e `compacidade`
(`4πA/P²`, borda serrilhada → valor baixo).

> ⚠️ **Sobre a referência.** A máscara de referência é um polígono traçado à mão sobre a imagem
> `socal` e é, ela própria, **aproximada** — na parte inferior da pluma não existe fronteira
> objetiva entre fumaça e *haze*. Portanto os valores de IoU/Dice servem para **comparar as
> configurações entre si**, não como medida absoluta de qualidade. Essa incerteza do rótulo é
> parte da resposta à pergunta "por que é difícil". Na imagem `yellowstone` não há referência,
> e as métricas sem referência continuam valendo.

In [ ]:
# =====================================================================
# ETAPA 5 — Métricas
# =====================================================================

def iou(a, b):
    u = np.logical_or(a, b).sum()
    return float(np.logical_and(a, b).sum() / u) if u else 0.0


def dice(a, b):
    s = a.sum() + b.sum()
    return float(2 * np.logical_and(a, b).sum() / s) if s else 0.0


def metricas(mask, rois, ref=None):
    m = mask.astype(bool)
    d = {"área (%)": 100 * m.mean()}
    n, rot, stats, _ = cv2.connectedComponentsWithStats(m.astype(np.uint8), 8)
    d["componentes"] = int(n - 1)
    d["maior comp. (%)"] = (100 * stats[1:, cv2.CC_STAT_AREA].max() / m.sum()
                            if n > 1 and m.sum() else 0.0)
    cont, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    per = sum(cv2.arcLength(c, True) for c in cont)
    d["compacidade"] = float(4 * np.pi * m.sum() / per ** 2) if per > 0 else 0.0
    for nome, roi in rois.items():
        rotulo = "cobertura pluma (%)" if nome == "pluma" else f"vazamento {nome} (%)"
        d[rotulo] = 100 * np.logical_and(m, roi).sum() / roi.sum()
    if ref is not None:
        vp = np.logical_and(m, ref).sum()
        d["IoU"] = iou(m, ref)
        d["Dice"] = dice(m, ref)
        d["precisão"] = float(vp / m.sum()) if m.sum() else 0.0
        d["recall"] = float(vp / ref.sum()) if ref.sum() else 0.0
    return d


TABELA = pd.DataFrame({nome: metricas(r["mascara"], ROIS, REFERENCIA)
                       for nome, r in RESULTADOS.items()}).T
TABELA.insert(0, "score SAM", [RESULTADOS[n]["score"] for n in TABELA.index])
TABELA.insert(1, "tempo (s)", [RESULTADOS[n]["segundos"] for n in TABELA.index])
TABELA = TABELA.round(3)
print("Tabela de avaliação (uma linha por configuração de prompt):")
display(TABELA)
TABELA.to_csv(os.path.join(PASTA_SAIDA, "05_metricas.csv"))

# ---- IoU par-a-par: o quanto a escolha do prompt muda a resposta ----
nomes = list(RESULTADOS)
M = np.array([[iou(RESULTADOS[a]["mascara"], RESULTADOS[b]["mascara"]) for b in nomes]
              for a in nomes])
fig, axs = plt.subplots(1, 2 if REFERENCIA is not None else 1,
                        figsize=(13 if REFERENCIA is not None else 6.5, 5))
axs = np.atleast_1d(axs)
im = axs[0].imshow(M, cmap="viridis", vmin=0, vmax=1)
axs[0].set_xticks(range(len(nomes)), nomes, rotation=45, ha="right", fontsize=7)
axs[0].set_yticks(range(len(nomes)), nomes, fontsize=7)
for i in range(len(nomes)):
    for j in range(len(nomes)):
        axs[0].text(j, i, f"{M[i,j]:.2f}", ha="center", va="center",
                    color="white" if M[i, j] < 0.6 else "black", fontsize=7)
axs[0].set_title("IoU entre as configurações\n(quanto menor, mais o prompt muda o resultado)")
plt.colorbar(im, ax=axs[0], fraction=0.046)

if REFERENCIA is not None:
    melhor_nome = TABELA["IoU"].idxmax()
    mm = RESULTADOS[melhor_nome]["mascara"]
    mapa = np.zeros((*mm.shape, 3), dtype=np.uint8)
    mapa[np.logical_and(mm, REFERENCIA)] = (60, 200, 60)        # acerto
    mapa[np.logical_and(mm, ~REFERENCIA)] = (230, 70, 70)       # falso positivo
    mapa[np.logical_and(~mm, REFERENCIA)] = (70, 120, 240)      # falso negativo
    axs[1].imshow((IMG * 0.45 + mapa * 0.55).astype(np.uint8))
    axs[1].set_title(f"Mapa de erro — {melhor_nome}\n"
                     f"verde = acerto | vermelho = incluiu o que não é pluma | azul = deixou fumaça de fora")
    axs[1].axis("off")
fig.tight_layout(); salvar_fig(fig, "05_metricas.png"); plt.show()

---
# Etapa 6 — O que exatamente dificultou: três experimentos controlados

A Etapa 2 mostrou que as regiões são estatisticamente parecidas. Aqui testamos se essa semelhança
**realmente** degrada a segmentação, variando um fator por vez.

1. **Robustez à caixa** — a caixa é perturbada aleatoriamente (±8% do lado) e mede-se o IoU contra a
   máscara original. Se cair muito, significa que o modelo está se apoiando na borda do retângulo em
   vez de numa fronteira de imagem — sintoma clássico de objeto com contraste fraco.
2. **Iluminação e contraste** — a mesma cena é reprocessada com gama e contraste alterados
   (sub e superexposição simuladas). Como o SAM recalcula o *embedding*, mudar a exposição muda o que
   ele enxerga; o IoU contra a condição original quantifica essa sensibilidade.
3. **Fumaça × nuvem** — um ponto positivo é colocado **na ROI de nuvem/haze**, mantendo o resto do
   *prompt*. O quanto a máscara cresce mede o quanto o modelo considera nuvem e fumaça a mesma coisa.
   O contraste entre as configurações C e D (sem e com negativos) mede o mesmo efeito pelo outro lado.

In [ ]:
# =====================================================================
# ETAPA 6 — Experimentos de sensibilidade
# =====================================================================
SENSIBILIDADE = {}
base = RESULTADOS["D — caixa + (+) + (−)"]["mascara"]
cfg_d = CONFIGS["D — caixa + (+) + (−)"]

if CFG["RODAR_SENSIBILIDADE"]:
    # ---------- 6.1 robustez à caixa ----------
    rng = np.random.default_rng(42)
    ious_jitter, caixas = [], []
    for _ in range(CFG["N_JITTER"]):
        dx, dy = (CAIXA_PX[2] - CAIXA_PX[0]) * 0.08, (CAIXA_PX[3] - CAIXA_PX[1]) * 0.08
        ruido = rng.uniform(-1, 1, 4) * np.array([dx, dy, dx, dy])
        c = np.clip(CAIXA_PX + ruido, [0, 0, 0, 0], [LARG, ALT, LARG, ALT])
        m, _, _, _, _ = prever(box=c, pontos=cfg_d["pontos"], rotulos=cfg_d["rotulos"])
        ious_jitter.append(iou(m, base)); caixas.append(c)
    SENSIBILIDADE["caixa"] = {"iou_medio": float(np.mean(ious_jitter)),
                              "desvio": float(np.std(ious_jitter)),
                              "pior": float(np.min(ious_jitter))}
    print(f"6.1 caixa ±8%   IoU médio={np.mean(ious_jitter):.3f} ± {np.std(ious_jitter):.3f} "
          f"(pior={np.min(ious_jitter):.3f})")

    # ---------- 6.2 iluminação e contraste ----------
    def ajustar(img, gama=1.0, contraste=1.0):
        x = img.astype(np.float32) / 255.0
        x = np.clip((x - 0.5) * contraste + 0.5, 0, 1) ** gama
        return (x * 255).astype(np.uint8)

    variantes = {"subexposta (γ=1.6)": dict(gama=1.6),
                 "superexposta (γ=0.6)": dict(gama=0.6),
                 "baixo contraste (×0.5)": dict(contraste=0.5)}
    SENSIBILIDADE["exposicao"] = {}
    figs_exp = []
    for nome_v, par in variantes.items():
        img_v = ajustar(IMG, **par)
        p2 = SamPredictor(sam)
        p2.set_image(img_v)
        m, s, _, _, _ = prever(box=cfg_d["box"], pontos=cfg_d["pontos"],
                               rotulos=cfg_d["rotulos"], pred=p2)
        SENSIBILIDADE["exposicao"][nome_v] = {"iou_vs_original": iou(m, base),
                                              "score": s,
                                              "delta_area_pp": 100 * (m.mean() - base.mean())}
        figs_exp.append((nome_v, img_v, m))
        print(f"6.2 {nome_v:24s} IoU vs original={iou(m, base):.3f}  "
              f"Δárea={100*(m.mean()-base.mean()):+.2f} p.p.")

    fig, axs = plt.subplots(1, len(figs_exp) + 1, figsize=(4.2 * (len(figs_exp) + 1), 3.6))
    axs[0].imshow(sobrepor(IMG, base)); axs[0].set_title("original (D)")
    for ax, (nome_v, img_v, m) in zip(axs[1:], figs_exp):
        ax.imshow(sobrepor(img_v, m))
        ax.set_title(f"{nome_v}\nIoU vs original = {iou(m, base):.2f}")
    for a in axs:
        a.axis("off")
    fig.suptitle("6.2 — mesma cena, mesmo prompt, exposição diferente", y=1.04)
    fig.tight_layout(); salvar_fig(fig, "06_exposicao.png"); plt.show()

    # ---------- 6.3 fumaça x nuvem ----------
    x0, y0, x1, y1 = caixa_para_px(PRESET["rois"]["nuvem_haze"], LARG, ALT)
    ponto_nuvem = np.array([[(x0 + x1) / 2, (y0 + y1) / 2]])
    pontos_nv = np.vstack([cfg_d["pontos"], ponto_nuvem])
    rotulos_nv = np.concatenate([cfg_d["rotulos"], [1]])
    m_nuvem, _, _, _, _ = prever(box=None, pontos=pontos_nv, rotulos=rotulos_nv)
    m_so_pos = RESULTADOS["C — caixa + (+)"]["mascara"]
    SENSIBILIDADE["nuvem"] = {
        "iou_com_ponto_na_nuvem_vs_D": iou(m_nuvem, base),
        "crescimento_area_pp": 100 * (m_nuvem.mean() - base.mean()),
        "iou_C_vs_D_efeito_dos_negativos": iou(m_so_pos, base),
        "delta_area_C_para_D_pp": 100 * (m_so_pos.mean() - base.mean()),
    }
    print(f"6.3 ponto + na nuvem/haze: IoU vs D={iou(m_nuvem, base):.3f}  "
          f"Δárea={100*(m_nuvem.mean()-base.mean()):+.2f} p.p.")
    print(f"6.3 efeito dos negativos (C vs D): IoU={iou(m_so_pos, base):.3f}  "
          f"Δárea={100*(m_so_pos.mean()-base.mean()):+.2f} p.p.")

    fig, axs = plt.subplots(1, 3, figsize=(16, 16 * ALT / LARG / 2.8))
    axs[0].imshow(sobrepor(IMG, base)); axs[0].set_title("D — com pontos negativos")
    axs[1].imshow(sobrepor(IMG, m_so_pos)); axs[1].set_title("C — sem pontos negativos")
    axs[2].imshow(sobrepor(IMG, m_nuvem))
    axs[2].scatter(ponto_nuvem[:, 0], ponto_nuvem[:, 1], c="cyan", marker="*", s=220, ec="k")
    axs[2].set_title("D + um ponto POSITIVO na nuvem/haze")
    for a in axs:
        a.axis("off")
    fig.suptitle("6.3 — a fronteira entre fumaça e nuvem/haze é decidida pelo prompt, não pela imagem", y=1.04)
    fig.tight_layout(); salvar_fig(fig, "06_nuvem_vs_fumaca.png"); plt.show()
else:
    print("Experimentos de sensibilidade desativados em CFG['RODAR_SENSIBILIDADE'].")

---
# Etapa 7 — Relatório automático

A célula abaixo **escreve o texto da avaliação a partir das métricas calculadas**, em vez de deixar
números soltos numa tabela. Serve para dois fins: responder às três perguntas do enunciado com os
valores da execução atual, e garantir que a conclusão da Etapa 8 esteja de acordo com o que de fato
saiu — se você trocar a imagem ou os *prompts*, este relatório muda junto.

In [ ]:
# =====================================================================
# ETAPA 7 — Relatório automático (texto gerado a partir dos números medidos)
# =====================================================================

def fmt(x, casas=2):
    return "—" if x is None or (isinstance(x, float) and np.isnan(x)) else f"{x:.{casas}f}"


def gerar_relatorio():
    L = []
    melhor_nome = TABELA["IoU"].idxmax() if "IoU" in TABELA else TABELA["cobertura pluma (%)"].idxmax()
    A, D = "A — só caixa", "D — caixa + (+) + (−)"
    L.append(f"IMAGEM: {CFG['IMAGEM']} ({LARG}x{ALT}) | modelo {TIPO_MODELO} em {DISPOSITIVO}")
    L.append("")
    L.append("1) A MÁSCARA COBRE A PLUMA?")
    L.append(f"   A melhor configuração foi '{melhor_nome}'.")
    if "recall" in TABELA:
        L.append(f"   Recall contra a referência: {fmt(TABELA.loc[melhor_nome,'recall'],3)} "
                 f"(deixou de fora {fmt(100*(1-TABELA.loc[melhor_nome,'recall']),1)}% da pluma de referência).")
    L.append(f"   Cobertura da ROI de pluma: {fmt(TABELA.loc[melhor_nome,'cobertura pluma (%)'],1)}%.")
    L.append("")
    L.append("2) INCLUI REGIÕES QUE NÃO SÃO PLUMA?")
    for col in [c for c in TABELA.columns if c.startswith("vazamento")]:
        pior = TABELA[col].idxmax()
        L.append(f"   {col:28s} melhor={fmt(TABELA[col].min(),1)}% ({TABELA[col].idxmin()}) | "
                 f"pior={fmt(TABELA[col].max(),1)}% ({pior})")
    if "precisão" in TABELA:
        L.append(f"   Precisão de '{melhor_nome}': {fmt(TABELA.loc[melhor_nome,'precisão'],3)}.")
    L.append("")
    L.append("3) EFEITO DE CADA ELEMENTO DO PROMPT")
    L.append(f"   Só caixa (A): área={fmt(TABELA.loc[A,'área (%)'])}% | "
             f"completo (D): área={fmt(TABELA.loc[D,'área (%)'])}%  "
             f"=> Δ={fmt(TABELA.loc[D,'área (%)']-TABELA.loc[A,'área (%)'])} p.p.")
    L.append(f"   IoU(A,D) = {fmt(iou(RESULTADOS[A]['mascara'], RESULTADOS[D]['mascara']),3)} "
             "— quanto menor, mais a resposta dependeu do prompt e não da imagem.")
    L.append(f"   Componentes conexos: A={int(TABELA.loc[A,'componentes'])}, "
             f"F={int(TABELA.loc['F — E + pós-proc.','componentes'])} (efeito do pós-processamento).")
    L.append("")
    L.append("4) POR QUE A IMAGEM É DIFÍCIL (medido na Etapa 2)")
    for comp in TAB_SEPARABILIDADE.index:
        vals = ", ".join(f"{c}={TAB_SEPARABILIDADE.loc[comp,c]:.2f}" for c in TAB_SEPARABILIDADE.columns)
        L.append(f"   Bhattacharyya {comp}: {vals}")
    try:
        d_nuvem = TAB_SEPARABILIDADE.loc["pluma × nuvem_haze"].mean()
        d_ceu = TAB_SEPARABILIDADE.loc["pluma × ceu"].mean()
        rel = "MENOR" if d_nuvem < d_ceu else "maior"
        L.append(f"   => a distância média pluma-nuvem ({d_nuvem:.2f}) é {rel} que pluma-céu ({d_ceu:.2f}): "
                 f"{'nuvem/haze é o distrator mais perigoso.' if d_nuvem < d_ceu else 'o céu ainda compete.'}")
    except KeyError:
        pass
    if SENSIBILIDADE:
        L.append("")
        L.append("5) SENSIBILIDADE (Etapa 6)")
        s = SENSIBILIDADE.get("caixa", {})
        if s:
            L.append(f"   Caixa perturbada em ±8%: IoU={fmt(s['iou_medio'],3)} ± {fmt(s['desvio'],3)} "
                     f"(pior caso {fmt(s['pior'],3)}).")
        for k, v in SENSIBILIDADE.get("exposicao", {}).items():
            L.append(f"   {k:24s}: IoU vs original={fmt(v['iou_vs_original'],3)}, "
                     f"Δárea={v['delta_area_pp']:+.2f} p.p.")
        n = SENSIBILIDADE.get("nuvem", {})
        if n:
            L.append(f"   Ponto positivo na nuvem/haze: IoU vs D={fmt(n['iou_com_ponto_na_nuvem_vs_D'],3)}, "
                     f"área {n['crescimento_area_pp']:+.2f} p.p.")
            L.append(f"   Sem pontos negativos (C vs D): IoU={fmt(n['iou_C_vs_D_efeito_dos_negativos'],3)}, "
                     f"área {n['delta_area_C_para_D_pp']:+.2f} p.p.")
    return "\n".join(L)


RELATORIO = gerar_relatorio()
print(RELATORIO)
with open(os.path.join(PASTA_SAIDA, "07_relatorio.txt"), "w", encoding="utf-8") as f:
    f.write(RELATORIO)

---
# Etapa 8 — Conclusões e justificativa das melhorias

*(Os números citados aqui são impressos pela Etapa 7 — rode o notebook e confira os valores da sua
execução antes de entregar; a leitura qualitativa abaixo vale para as duas imagens do catálogo.)*

## 8.1 O que o SAM acertou e o que errou

O SAM é **agnóstico a classe**: ele não tem o conceito "fumaça". Ele resolve "qual região coerente
o *prompt* está indicando". Isso explica os três comportamentos observados:

* **Modo automático (Etapa 1)** — a pluma nunca sai como um objeto único. Ela aparece fatiada em
  várias máscaras, e céu e telhados recebem máscaras próprias. Sem *prompt* não há como o modelo
  saber que *aquela* região é a que interessa.
* **Só caixa (A)** — a caixa que contém a pluma contém também muito céu e, em `socal`, parte do
  horizonte. O SAM tende a devolver a região mais contrastada dentro do retângulo, o que leva a
  máscara a vazar para o céu ou, ao contrário, a encolher para o núcleo denso da pluma. É o caso em
  que `multimask_output` entrega três respostas igualmente defensáveis.
* **Só pontos positivos (B)** — sem limite espacial, a máscara cresce por continuidade de aparência
  e engole a faixa de *haze* do horizonte, que tem exatamente a mesma estatística da base da pluma.

## 8.2 As melhorias, em ordem de impacto

1. **Pontos negativos (C → D) — maior ganho isolado.** Céu, nuvem/haze, telhado e placa recebem
   marcação explícita de "não é isto". O efeito aparece na queda das colunas `vazamento *` da tabela
   e no experimento 6.3. *Por quê:* o decodificador do SAM é condicionado nos *prompts*; um ponto
   negativo empurra a fronteira de decisão para fora daquela região, coisa que nenhuma mudança de
   caixa consegue fazer com a mesma precisão.
2. **Caixa + positivos juntos (C).** A caixa resolve *onde*; os positivos resolvem *em que escala*.
   Sozinho, cada um falha de um jeito diferente — e os dois erros são visíveis nas linhas A e B.
3. **Refinamento iterativo (E).** Reinjetar os *logits* em `mask_input` com `multimask_output=False`
   usa a predição anterior como *prior* e tipicamente estabiliza o contorno, reduzindo a oscilação
   da borda. É barato: não recalcula o *embedding*, só o decodificador leve.
4. **Pós-processamento morfológico (F).** Abertura remove respingos isolados; fechamento costura
   fissuras finas; manter o maior componente conexo elimina ilhas de céu classificadas por engano.
   *Cuidado honesto:* se a pluma for genuinamente descontínua (ocluída pela placa, em `socal`), manter
   só o maior componente **descarta fumaça real** — por isso a função tem `manter_maior=False` como
   alternativa, e a escolha deve ser justificada caso a caso, não aplicada no automático.
5. **Máscara suave (apresentação).** O filtro guiado converte a borda dura numa faixa de transição
   alinhada às bordas da imagem. Isso não "melhora o IoU" — melhora a **honestidade** da
   apresentação, porque fumaça é um meio semitransparente e o limiar binário é uma decisão nossa.

## 8.3 Respondendo diretamente: o que dificultou a segmentação

* **Semelhança fumaça × nuvem/haze.** É o fator dominante. Na Etapa 2 a distância de Bhattacharyya
  entre pluma e nuvem/haze fica bem abaixo da distância entre pluma e céu em luminância e saturação:
  as duas classes se sobrepõem no histograma. Na prática, a fronteira é definida pelo *prompt*
  (experimento 6.3), não pela imagem — mover um ponto muda a resposta.
* **Contraste e borda difusa.** O mapa de gradiente mostra bordas fortes em telhados, poste e
  cordilheira, e borda fraca na pluma. Modelos de segmentação ancoram o contorno em gradiente; sem
  gradiente, a borda fica arbitrária, e é por isso que a caixa perturbada em ±8% (6.1) já muda o
  resultado — o modelo se apoia na borda do retângulo quando a imagem não oferece uma fronteira.
* **Iluminação.** O experimento 6.2 reprocessa a mesma cena com gama/contraste alterados. Como o
  *embedding* é recalculado, a máscara muda sem que nada na cena tenha mudado. Isso diz, na prática,
  que fotos do mesmo incêndio em horários diferentes não são diretamente comparáveis sem normalização.
* **Oclusão e contexto urbano** (`socal`). A placa de trânsito corta a pluma em duas; sem um ponto
  negativo sobre ela, a placa é absorvida pela máscara. Telhados e encosta sob a pluma competem na
  parte inferior da caixa.
* **Transparência.** Perto da borda, cada pixel é mistura de fumaça e fundo. Nenhuma máscara binária
  é "correta" ali — é um limite do formato de saída, não do modelo.

## 8.4 O que eu faria a seguir

* Rodar o mesmo protocolo em **N imagens** e reportar média ± desvio, em vez de um caso só.
* Testar o **SAM 2**, cujo decodificador trata melhor objetos de fronteira difusa, e comparar com
  as mesmas métricas desta tabela.
* Usar o **canal de transparência** como saída oficial (estimativa de alfa em vez de máscara
  binária), que é o formato adequado para quantificar área de pluma em séries temporais.
* Marcar uma **referência de consenso** (duas ou três pessoas traçando a mesma pluma) e reportar a
  concordância entre elas — ela é o teto realista de IoU que qualquer modelo pode atingir aqui.

---
# Etapa 9 — Exportar os resultados

Salva figuras, máscaras em PNG, o CSV de métricas e o relatório em `saida/`, e baixa tudo num `.zip`.
Útil para anexar imagens no relatório escrito sem precisar reexecutar o notebook.

In [ ]:
# =====================================================================
# ETAPA 9 — Exportar resultados (figuras, máscaras e tabela)
# =====================================================================
import shutil

# máscaras em PNG, prontas para anexar no relatório
for nome, r in RESULTADOS.items():
    rotulo = nome.split(" ")[0]
    cv2.imwrite(os.path.join(PASTA_SAIDA, f"mascara_{rotulo}.png"),
                (r["mascara"] * 255).astype(np.uint8))
    cv2.imwrite(os.path.join(PASTA_SAIDA, f"sobreposicao_{rotulo}.png"),
                cv2.cvtColor(sobrepor(IMG, r["mascara"]), cv2.COLOR_RGB2BGR))
cv2.imwrite(os.path.join(PASTA_SAIDA, "mascara_suave.png"),
            (MASCARA_SUAVE * 255).astype(np.uint8))

shutil.make_archive("resultados_sam", "zip", PASTA_SAIDA)
print("Arquivos gerados em", PASTA_SAIDA, ":")
for f in sorted(os.listdir(PASTA_SAIDA)):
    print("  -", f)

try:
    from google.colab import files
    files.download("resultados_sam.zip")
except Exception as e:
    print("(download automático indisponível fora do Colab)", e)

---
# Etapa 10 — Como entregar (link do GitHub)

### Opção 1 — direto do Colab (mais rápida)

1. `Arquivo` → `Salvar uma cópia no GitHub`.
2. Autorize o Colab a acessar sua conta do GitHub.
3. Escolha o repositório (ou crie um, por exemplo `segmentacao-pluma-sam`), marque
   **"Incluir um link para o Colab"** e confirme.
4. O link a entregar é o do arquivo `.ipynb` no GitHub, no formato
   `https://github.com/SEU_USUARIO/SEU_REPO/blob/main/Segmentacao_Pluma_SAM.ipynb`.

> Dica: rode **tudo** antes de salvar, para que o notebook no GitHub já apareça com as figuras.
> O GitHub renderiza `.ipynb` com as saídas embutidas — é assim que o professor vê os resultados
> sem precisar executar nada.

### Opção 2 — upload manual

1. Crie o repositório em <https://github.com/new>.
2. `Arquivo` → `Fazer o download` → `Fazer o download do .ipynb`.
3. No repositório, `Add file` → `Upload files`, envie o `.ipynb` (e, se quiser, o `resultados_sam.zip`).

### Opção 3 — linha de comando

```bash
git init
git add Segmentacao_Pluma_SAM.ipynb README.md
git commit -m "Segmentacao de pluma de fumaca com SAM"
git branch -M main
git remote add origin https://github.com/SEU_USUARIO/SEU_REPO.git
git push -u origin main
```

### Checklist antes de entregar

- [ ] O notebook foi executado do começo ao fim e as **saídas estão salvas** no `.ipynb`.
- [ ] As figuras da Etapa 4 mostram original, binária e sobreposição para **cada** configuração.
- [ ] A tabela da Etapa 5 aparece preenchida.
- [ ] O relatório da Etapa 7 foi impresso com os números da **sua** execução.
- [ ] A Etapa 8 foi revisada por você — ajuste as frases aos números que de fato saíram.
- [ ] O repositório é **público** (ou o professor tem acesso).